# Configure Elyra Runtime and Runtime Images for Air-Gapped KFP

This notebook follows the `00Visual development environment configuration.ipynb` workflow. Run it from a JupyterLab Workbench that already contains Elyra and the KFP SDK. It writes Elyra metadata into the Workbench user's Jupyter data directory; it does not create or mount a PVC, ConfigMap, or WorkspaceKind.

For a disconnected cluster, use cluster-internal KFP and object-storage endpoints and mirror every pipeline runtime image into a registry reachable by the KFP workload nodes. The notebook never pulls images or Python packages.

## Prerequisites

- The Workbench image includes `elyra-metadata`, Elyra, and the KFP SDK.
- KFP and its artifact object store are installed in the target cluster.
- The KFP API and object-storage endpoints below are reachable from the Workbench without internet access.
- The five pipeline runtime images have been copied to the internal registry. If that registry requires authentication, create an image pull Secret in the KFP namespace and set `ELYRA_RUNTIME_PULL_SECRET`.

On a connected staging host, a multi-architecture image can be mirrored with `skopeo copy --all docker://alaudadockerhub/<image>:<tag> docker://<internal-registry>/<project>/<image>:<tag>`. Repeat this for every runtime image and for the JupyterLab Workbench image, then make the internal registry content available in the disconnected environment.

Set the variables in the next cell, or export the matching environment variables before running this notebook. Do not commit real object-storage credentials to the notebook.

In [ ]:
import os
import shutil

# Replace the placeholder values, or set these variables in the Workbench environment.
NAMESPACE = os.getenv("ELYRA_KFP_NAMESPACE", "<your-namespace>")
KFP_API_ENDPOINT = os.getenv("ELYRA_KFP_API_ENDPOINT", "http://ml-pipeline.kubeflow:8888")
COS_ENDPOINT = os.getenv("ELYRA_COS_ENDPOINT", "http://minio-service.kubeflow:9000")
COS_BUCKET = os.getenv("ELYRA_COS_BUCKET", "<your-kfp-artifact-bucket>")
COS_USERNAME = os.getenv("ELYRA_COS_USERNAME", "<object-storage-access-key>")
COS_PASSWORD = os.getenv("ELYRA_COS_PASSWORD", "<object-storage-secret-key>")
IMAGE_REGISTRY = os.getenv("ELYRA_RUNTIME_REGISTRY", "<internal-registry>/<project>")
IMAGE_TAG = os.getenv("ELYRA_RUNTIME_TAG", "<tag>")
IMAGE_PULL_SECRET = os.getenv("ELYRA_RUNTIME_PULL_SECRET", "")

required_values = {
    "ELYRA_KFP_NAMESPACE": NAMESPACE,
    "ELYRA_KFP_API_ENDPOINT": KFP_API_ENDPOINT,
    "ELYRA_COS_ENDPOINT": COS_ENDPOINT,
    "ELYRA_COS_BUCKET": COS_BUCKET,
    "ELYRA_COS_USERNAME": COS_USERNAME,
    "ELYRA_COS_PASSWORD": COS_PASSWORD,
    "ELYRA_RUNTIME_REGISTRY": IMAGE_REGISTRY,
    "ELYRA_RUNTIME_TAG": IMAGE_TAG,
}
missing = [name for name, value in required_values.items() if not value or value.startswith("<")]
if missing:
    raise ValueError("Set these values before continuing: " + ", ".join(missing))
if shutil.which("elyra-metadata") is None:
    raise RuntimeError("elyra-metadata is not installed in this Workbench image")

print(f"KFP endpoint: {KFP_API_ENDPOINT}")
print(f"Object storage endpoint: {COS_ENDPOINT}")
print(f"Runtime image prefix: {IMAGE_REGISTRY}")
print("Configuration values are ready.")

In [ ]:
import subprocess

def run_metadata(*args):
    command = ["elyra-metadata", *args]
    display_command = list(command)
    for secret_flag in ("--cos_password",):
        if secret_flag in display_command:
            display_command[display_command.index(secret_flag) + 1] = "***"
    print("$", display_command)
    subprocess.run(command, check=True)

runtime_args = [
    "create",
    "runtimes",
    "--schema_name",
    "kfp",
    "--display_name",
    "MLOps KFP (air-gapped)",
    "--api_endpoint",
    KFP_API_ENDPOINT,
    "--user_namespace",
    NAMESPACE,
    "--auth_type",
    "KUBERNETES_SERVICE_ACCOUNT_TOKEN",
    "--engine",
    "Argo",
    "--cos_endpoint",
    COS_ENDPOINT,
    "--cos_auth_type",
    "USER_CREDENTIALS",
    "--cos_username",
    COS_USERNAME,
    "--cos_password",
    COS_PASSWORD,
    "--cos_bucket",
    COS_BUCKET,
    "--tags",
    "['kfp', 'air-gapped']",
]
run_metadata(*runtime_args)
print("KFP Runtime metadata created.")

In [ ]:
runtime_images = [
    ("odh-pipeline-runtime-minimal-cpu-py312-ubi9", "Runtime | Minimal | CPU | Python 3.12", "Minimal runtime image for Elyra pipeline nodes."),
    ("odh-pipeline-runtime-datascience-cpu-py312-ubi9", "Runtime | Data Science | CPU | Python 3.12", "Data science runtime image for Elyra pipeline nodes."),
    ("odh-pipeline-runtime-tensorflow-cuda-py312-ubi9", "Runtime | TensorFlow | CUDA | Python 3.12", "TensorFlow CUDA runtime image for Elyra pipeline nodes."),
    ("odh-pipeline-runtime-pytorch-cuda-py312-ubi9", "Runtime | PyTorch | CUDA | Python 3.12", "PyTorch CUDA runtime image for Elyra pipeline nodes."),
    ("odh-pipeline-runtime-pytorch-llmcompressor-cuda-py312-ubi9", "Runtime | PyTorch LLM Compressor | CUDA | Python 3.12", "PyTorch and LLM Compressor CUDA runtime image for Elyra pipeline nodes."),
]

for name, display_name, description in runtime_images:
    image_args = [
        "create",
        "runtime-images",
        "--name",
        name,
        "--display_name",
        display_name,
        "--description",
        description,
        "--image_name",
        f"{IMAGE_REGISTRY}/{name}:{IMAGE_TAG}",
        "--pull_policy",
        "IfNotPresent",
    ]
    if IMAGE_PULL_SECRET and not IMAGE_PULL_SECRET.startswith("<"):
        image_args.extend(["--pull_secret", IMAGE_PULL_SECRET])
    run_metadata(*image_args)

print(f"Created {len(runtime_images)} air-gapped Runtime Image entries.")

In [ ]:
# Verify the metadata that Elyra will read from this Workbench.
run_metadata("list", "runtimes")
run_metadata("list", "runtime-images")

## Rerunning the notebook

The `create` commands fail when an instance with the same name already exists. Keep the existing metadata and use `elyra-metadata update` with the same arguments when changing an endpoint or image. Do not remove metadata unless you intentionally want to delete that Runtime or Runtime Image from this Workbench.

After creation, refresh JupyterLab and select the new KFP Runtime and Runtime Image entries in the Elyra pipeline editor.